# ConvNeXt-Tiny-IN22K Baseline, TFLite Export, and TFLite Evaluation

This notebook is a focused follow-up to `training_and_log_v2.ipynb` and `mobile_accuracy_optimization_effb0_yolo26n.ipynb`.

It does one experiment:

1. Load the same processed shrimp dataset from `Processed_Rembg_Images`.
2. Recreate the same fixed stratified `70/15/15` train/validation/test split with seed `42`.
3. Train `convnext_tiny_in22k` as a PyTorch/timm classifier.
4. Evaluate the best validation checkpoint on the held-out test set in PyTorch.
5. Export the best checkpoint to ONNX.
6. Convert ONNX to TensorFlow SavedModel and TFLite.
7. Evaluate the exported TFLite model on the same held-out test split.

The final deployment-facing numbers are the TFLite test metrics.

In [ ]:
# Colab dependencies. Restart the runtime if TensorFlow/onnx2tf packages are changed by pip.
%pip -q install timm torchmetrics onnx onnxruntime onnxslim onnxsim onnxscript tensorflow==2.19.1 tf-keras==2.19.0 ai-edge-litert onnx2tf==1.28.8 onnx-graphsurgeon sng4onnx pandas pillow matplotlib seaborn tqdm scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

The split, class order, image size, and ImageNet normalization match the previous baseline notebooks. Outputs are isolated under a new Drive folder.

In [ ]:
import gc
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchmetrics
import timm
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

try:
    from ai_edge_litert.interpreter import Interpreter
    INTERPRETER_BACKEND = 'ai_edge_litert'
except Exception:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
    INTERPRETER_BACKEND = 'tensorflow.lite'

import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Using TFLite interpreter backend: {INTERPRETER_BACKEND}')

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
PATIENCE = 3
LR = 1e-4
NUM_WORKERS = 2

DATA_DIR = Path('/content/drive/MyDrive/shrimp_disease_images/Processed_Rembg_Images')
OUTPUT_DIR = Path('/content/drive/MyDrive/shrimp_disease_images/convnext_tiny_in22k_baseline')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
EXPORT_DIR = OUTPUT_DIR / 'exports'
SAVED_MODEL_DIR = OUTPUT_DIR / 'saved_models'
TFLITE_DIR = OUTPUT_DIR / 'tflite'
REPORT_DIR = OUTPUT_DIR / 'reports'
for d in [CHECKPOINT_DIR, EXPORT_DIR, SAVED_MODEL_DIR, TFLITE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_DIRS = ['1. Healthy', '2. BG', '3. WSSV', '4. WSSV_BG']
CLASS_NAMES = ['Healthy', 'BG', 'WSSV', 'WSSV_BG']
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
NUM_CLASSES = len(CLASS_NAMES)

# timm model name for ConvNeXt-Tiny pretrained on ImageNet-22K.
# If this name is unavailable in a future timm build, inspect timm.list_models('convnext_tiny*', pretrained=True).
TIMM_MODEL_NAME = 'convnext_tiny.fb_in22k'
MODEL_KEY = 'convnext_tiny_in22k'

# Keep float32 as the primary paper/baseline-style export. Add 'float16' if you also want a smaller mobile variant.
TFLITE_VARIANTS = ['float32']

print(f'Output root: {OUTPUT_DIR}')

In [ ]:
def discover_processed_images(data_dir: Path) -> pd.DataFrame:
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        if not folder.exists():
            print(f'Warning: missing class folder: {folder}')
            continue
        for p in sorted(folder.iterdir()):
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                rows.append({'path': str(p), 'class_dir': class_dir, 'label': CLASS_TO_IDX[class_dir]})
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No images found under {data_dir}. Run the preprocessing cell first.')
    return df

df = discover_processed_images(DATA_DIR)
print(f'Loaded {len(df)} processed images from {DATA_DIR}')
print(df['class_dir'].value_counts().reindex(CLASS_DIRS))

train_df, tmp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=SEED,
    shuffle=True,
)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=0.50,
    stratify=tmp_df['label'],
    random_state=SEED,
    shuffle=True,
)

for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: {len(split_df)} images')
    print(split_df['class_dir'].value_counts().reindex(CLASS_DIRS).to_dict())

train_paths = set(train_df['path'])
val_paths = set(val_df['path'])
test_paths = set(test_df['path'])
assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)
print('Image-level split overlap check passed.')

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

torch_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])

class ShrimpDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label'])

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    ShrimpDataset(train_df, torch_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)
val_loader = DataLoader(ShrimpDataset(val_df, torch_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(ShrimpDataset(test_df, torch_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

## Train ConvNeXt-Tiny-IN22K

The model is loaded from `timm` with ImageNet-22K pretrained weights and a new 4-class classifier head. Checkpoint selection uses validation macro F1, matching the baseline convention.

In [ ]:
def macro_f1_metric():
    return torchmetrics.F1Score(task='multiclass', num_classes=NUM_CLASSES, average='macro').to(device)

def count_params(model) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6

def build_convnext_tiny_in22k():
    model = timm.create_model(TIMM_MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
    return model.to(device)

def evaluate_pytorch_model(model, loader, criterion=None, timed=False):
    model.eval()
    f1 = macro_f1_metric()
    correct = 0
    total = 0
    total_loss = 0.0
    y_true = []
    y_pred = []

    if timed:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
        with torch.no_grad():
            for _ in range(3):
                model(dummy)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        start = time.time()
    else:
        start = None

    with torch.no_grad():
        for ims, gts in loader:
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            logits = model(ims)
            if criterion is not None:
                total_loss += criterion(logits, gts).item() * ims.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == gts).sum().item()
            total += gts.size(0)
            f1.update(logits, gts)
            y_true.extend(gts.detach().cpu().numpy().tolist())
            y_pred.extend(preds.detach().cpu().numpy().tolist())

    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else None

    true_names = [CLASS_NAMES[i] for i in y_true]
    pred_names = [CLASS_NAMES[i] for i in y_pred]
    return {
        'loss': total_loss / max(1, total) if criterion is not None else None,
        'accuracy': correct / max(1, total),
        'macro_f1': f1.compute().item(),
        'cohen_kappa': cohen_kappa_score(true_names, pred_names, labels=CLASS_NAMES),
        'elapsed': elapsed,
        'y_true': true_names,
        'y_pred': pred_names,
    }

def train_convnext():
    model = build_convnext_tiny_in22k()
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    best_f1 = -1.0
    epochs_no_improve = 0
    best_path = CHECKPOINT_DIR / f'best_{MODEL_KEY}.pth'
    history = []
    train_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for ims, gts in tqdm(train_loader, desc=f'{MODEL_KEY} epoch {epoch + 1}/{EPOCHS}', leave=False):
            ims = ims.to(device, non_blocking=True)
            gts = gts.to(device, non_blocking=True)
            logits = model(ims)
            loss = criterion(logits, gts)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * ims.size(0)
            train_correct += (torch.argmax(logits, dim=1) == gts).sum().item()
            train_total += gts.size(0)

        val_metrics = evaluate_pytorch_model(model, val_loader, criterion=criterion)
        train_acc = train_correct / max(1, train_total)
        epoch_record = {
            'epoch': epoch + 1,
            'train_loss': train_loss / max(1, train_total),
            'train_accuracy': train_acc,
            'val_loss': val_metrics['loss'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_accuracy': val_metrics['accuracy'],
        }
        history.append(epoch_record)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {epoch_record['train_loss']:.4f} - Acc: {train_acc:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} - Macro F1: {val_metrics['macro_f1']:.4f}"
        )

        if val_metrics['macro_f1'] > best_f1:
            best_f1 = val_metrics['macro_f1']
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            print(f'  --> Saved best checkpoint: macro F1 {best_f1:.4f}')
        else:
            epochs_no_improve += 1
            print(f'  --> No improvement ({epochs_no_improve}/{PATIENCE})')

        if epochs_no_improve >= PATIENCE:
            print('  --> Early stopping triggered.')
            break

    training_time = time.time() - train_start
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    test_metrics = evaluate_pytorch_model(model, test_loader, timed=True)
    history_df = pd.DataFrame(history)
    history_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_training_history.csv', index=False)

    train_record = {
        'model_key': MODEL_KEY,
        'timm_model_name': TIMM_MODEL_NAME,
        'parameters_m': round(count_params(model), 2),
        'checkpoint_path': str(best_path),
        'training_time_s': round(training_time, 1),
        'best_val_macro_f1': best_f1,
        'pytorch_test_accuracy': test_metrics['accuracy'],
        'pytorch_test_macro_f1': test_metrics['macro_f1'],
        'pytorch_test_cohen_kappa': test_metrics['cohen_kappa'],
        'pytorch_test_latency_ms': (test_metrics['elapsed'] / len(test_df)) * 1000,
    }
    pd.DataFrame([train_record]).to_csv(REPORT_DIR / f'{MODEL_KEY}_pytorch_test_summary.csv', index=False)
    return model, train_record, test_metrics

convnext_model, pytorch_summary, pytorch_test_metrics = train_convnext()
display(pd.DataFrame([pytorch_summary]))

## Export ConvNeXt to ONNX, SavedModel, and TFLite

The baseline PyTorch-to-TFLite direct path previously fell back to ONNX. This notebook uses the proven path from the conversion notebook: PyTorch ? ONNX ? `onnx2tf` SavedModel ? TFLite.

In [ ]:
def run_command(cmd, cwd=None, check=True):
    print('Running:', ' '.join(map(str, cmd)))
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout[-4000:])
    if completed.stderr:
        print(completed.stderr[-4000:])
    if check and completed.returncode != 0:
        message = (
            f'Command failed with exit code {completed.returncode}: {cmd}\n'
            f'STDOUT tail:\n{completed.stdout[-4000:]}\n'
            f'STDERR tail:\n{completed.stderr[-4000:]}'
        )
        raise RuntimeError(message)
    return completed

def export_to_onnx(model):
    model_cpu = model.to('cpu').eval()
    sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    onnx_path = EXPORT_DIR / f'{MODEL_KEY}.onnx'
    torch.onnx.export(
        model_cpu,
        sample,
        str(onnx_path),
        input_names=['input'],
        output_names=['logits'],
        opset_version=17,
        dynamo=False,
    )
    model.to(device)
    print(f'Saved ONNX: {onnx_path} ({onnx_path.stat().st_size / (1024 * 1024):.2f} MB)')
    return onnx_path

def find_saved_model_dir(output_dir: Path):
    candidates = [p for p in output_dir.rglob('saved_model.pb')]
    if (output_dir / 'saved_model.pb').exists():
        return output_dir
    if candidates:
        return candidates[0].parent
    raise FileNotFoundError(f'No saved_model.pb found under {output_dir}')

def clean_onnx2tf_sample_artifacts():
    # onnx2tf may download a sample .npy file for signature/validation helpers.
    # If Colab/Drive leaves a corrupt partial file, np.load can fail with
    # "Cannot load file containing pickled data when allow_pickle=False".
    patterns = [
        'calibration_image_sample_data*.npy',
        'calibration_image_sample_data*.npy.zip',
        'test_image_data*.npy',
    ]
    roots = [Path.cwd(), Path('/content'), EXPORT_DIR, SAVED_MODEL_DIR]
    removed = []
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for file_path in root.glob(pattern):
                try:
                    file_path.unlink()
                    removed.append(str(file_path))
                except Exception:
                    pass
    if removed:
        print('Removed stale onnx2tf sample artifacts:')
        for item in removed:
            print(' ', item)

def convert_onnx_to_saved_model(onnx_path):
    out_dir = SAVED_MODEL_DIR / onnx_path.stem

    def attempt(extra_args):
        if out_dir.exists():
            shutil.rmtree(out_dir)
        cmd = [
            sys.executable, '-m', 'onnx2tf',
            '-i', str(onnx_path),
            '-o', str(out_dir),
            *extra_args,
        ]
        return run_command(cmd, check=False)

    first = attempt(['-osd'])
    combined_log = (first.stdout or '') + '\n' + (first.stderr or '')
    if first.returncode != 0:
        if 'Cannot load file containing pickled data' in combined_log:
            print('onnx2tf failed while loading its sample image data. Retrying without -osd or -b after cleaning stale sample artifacts.')
            clean_onnx2tf_sample_artifacts()
            second = attempt([])
            if second.returncode != 0:
                raise RuntimeError(
                    f'onnx2tf retry without -osd/-b failed with exit code {second.returncode}.\n'
                    f'STDOUT tail:\n{second.stdout[-4000:]}\nSTDERR tail:\n{second.stderr[-4000:]}'
                )
        else:
            raise RuntimeError(
                f'onnx2tf failed with exit code {first.returncode}.\n'
                f'STDOUT tail:\n{first.stdout[-4000:]}\nSTDERR tail:\n{first.stderr[-4000:]}'
            )

    saved_model_dir = find_saved_model_dir(out_dir)
    print(f'Saved TensorFlow SavedModel: {saved_model_dir}')
    return saved_model_dir

def convert_saved_model_to_tflite(saved_model_path, variant='float32'):
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_path))
    if variant == 'float32':
        pass
    elif variant == 'dynamic_range':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif variant == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    else:
        raise ValueError(f'Unsupported TFLite variant: {variant}')

    tflite_bytes = converter.convert()
    out_path = TFLITE_DIR / f'{MODEL_KEY}_{variant}.tflite'
    out_path.write_bytes(tflite_bytes)
    print(f'Saved {variant} TFLite: {out_path} ({out_path.stat().st_size / (1024 * 1024):.2f} MB)')
    return out_path

onnx_path = export_to_onnx(convnext_model)
saved_model_path = convert_onnx_to_saved_model(onnx_path)

conversion_records = []
for variant in TFLITE_VARIANTS:
    try:
        t0 = time.time()
        tflite_path = convert_saved_model_to_tflite(saved_model_path, variant=variant)
        conversion_records.append({
            'model_key': f'{MODEL_KEY}_{variant}',
            'variant': variant,
            'status': 'converted',
            'path': str(tflite_path),
            'size_mb': round(tflite_path.stat().st_size / (1024 * 1024), 2),
            'seconds': round(time.time() - t0, 1),
            'error': '',
        })
    except Exception as exc:
        conversion_records.append({
            'model_key': f'{MODEL_KEY}_{variant}',
            'variant': variant,
            'status': 'failed',
            'path': '',
            'size_mb': np.nan,
            'seconds': np.nan,
            'error': f'{type(exc).__name__}: {exc}',
        })

conversion_df = pd.DataFrame(conversion_records)
conversion_df.to_csv(REPORT_DIR / f'{MODEL_KEY}_tflite_conversion_report.csv', index=False)
display(conversion_df)

## Evaluate TFLite on the Held-Out Test Split

This is the deployment-facing evaluation. The TFLite interpreter uses the same RGB resize + ImageNet normalization expected by the PyTorch ConvNeXt model.

In [ ]:
def load_interpreter(model_path):
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    return interpreter, interpreter.get_input_details()[0], interpreter.get_output_details()[0]

def infer_hw_from_input_shape(input_shape):
    shape = [int(x) for x in input_shape]
    if len(shape) != 4:
        return IMG_SIZE, IMG_SIZE, 'NHWC'
    if shape[1] == 3:
        return shape[2], shape[3], 'NCHW'
    return shape[1], shape[2], 'NHWC'

def preprocess_for_tflite(image_path, input_details):
    input_shape = input_details['shape']
    input_dtype = input_details['dtype']
    height, width, layout = infer_hw_from_input_shape(input_shape)

    image = Image.open(image_path).convert('RGB').resize((width, height))
    x = np.asarray(image, dtype=np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    if layout == 'NCHW':
        x = np.transpose(x, (2, 0, 1))
    x = np.expand_dims(x, axis=0)

    if np.issubdtype(input_dtype, np.integer):
        scale, zero_point = input_details.get('quantization', (0.0, 0))
        if scale and scale > 0:
            x = x / scale + zero_point
        info = np.iinfo(input_dtype)
        x = np.clip(np.rint(x), info.min, info.max).astype(input_dtype)
    else:
        x = x.astype(input_dtype)
    return x

def dequantize_output(y, output_details):
    y = np.asarray(y)
    if np.issubdtype(y.dtype, np.integer):
        scale, zero_point = output_details.get('quantization', (0.0, 0))
        if scale and scale > 0:
            y = (y.astype(np.float32) - zero_point) * scale
    return y

def as_probabilities(output):
    values = np.asarray(output).reshape(-1).astype(np.float64)
    if values.size == len(CLASS_NAMES) and np.all(values >= 0) and np.isclose(values.sum(), 1.0, atol=1e-3):
        return values.astype(np.float32)
    values = values - np.max(values)
    exp_values = np.exp(values)
    return (exp_values / exp_values.sum()).astype(np.float32)

def evaluate_tflite_model(model_key, model_path, eval_df):
    interpreter, input_details, output_details = load_interpreter(model_path)

    warmup_x = preprocess_for_tflite(eval_df.iloc[0]['path'], input_details)
    interpreter.set_tensor(input_details['index'], warmup_x)
    interpreter.invoke()

    rows = []
    start = time.perf_counter()
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=model_key, leave=False):
        x = preprocess_for_tflite(row['path'], input_details)
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details['index'], x)
        interpreter.invoke()
        latency_ms = (time.perf_counter() - t0) * 1000
        raw_output = dequantize_output(interpreter.get_tensor(output_details['index']), output_details)
        probs = as_probabilities(raw_output)
        pred_idx = int(np.argmax(probs))
        record = {
            'model_key': model_key,
            'image_path': row['path'],
            'true_label': CLASS_NAMES[int(row['label'])],
            'predicted_label': CLASS_NAMES[pred_idx],
            'confidence': float(probs[pred_idx]),
            'latency_ms': latency_ms,
        }
        for class_name, prob in zip(CLASS_NAMES, probs):
            record[f'prob_{class_name}'] = float(prob)
        rows.append(record)
    elapsed = time.perf_counter() - start

    pred_df = pd.DataFrame(rows)
    summary = {
        'model_key': model_key,
        'n_images': len(pred_df),
        'accuracy': accuracy_score(pred_df['true_label'], pred_df['predicted_label']),
        'macro_f1': f1_score(pred_df['true_label'], pred_df['predicted_label'], labels=CLASS_NAMES, average='macro', zero_division=0),
        'cohen_kappa': cohen_kappa_score(pred_df['true_label'], pred_df['predicted_label'], labels=CLASS_NAMES),
        'total_inference_time_s': elapsed,
        'mean_latency_ms': pred_df['latency_ms'].mean(),
        'fps': len(pred_df) / elapsed if elapsed > 0 else np.nan,
        'model_size_mb': round(Path(model_path).stat().st_size / (1024 * 1024), 2),
        'path': str(model_path),
    }
    return summary, pred_df

summaries = []
prediction_frames = []
for _, row in conversion_df[conversion_df['status'] == 'converted'].iterrows():
    summary, pred_df = evaluate_tflite_model(row['model_key'], Path(row['path']), test_df)
    summaries.append(summary)
    prediction_frames.append(pred_df)

tflite_summary_df = pd.DataFrame(summaries).sort_values('macro_f1', ascending=False).reset_index(drop=True)
tflite_predictions_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()

tflite_summary_path = REPORT_DIR / f'{MODEL_KEY}_tflite_test_summary.csv'
tflite_predictions_path = REPORT_DIR / f'{MODEL_KEY}_tflite_test_predictions.csv'
tflite_summary_df.to_csv(tflite_summary_path, index=False)
if not tflite_predictions_df.empty:
    tflite_predictions_df.to_csv(tflite_predictions_path, index=False)

print(f'Saved TFLite summary: {tflite_summary_path}')
print(f'Saved TFLite predictions: {tflite_predictions_path}')
display(tflite_summary_df)

In [ ]:
per_class_rows = []
for model_key, group in tflite_predictions_df.groupby('model_key'):
    report = classification_report(
        group['true_label'],
        group['predicted_label'],
        labels=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    for class_name in CLASS_NAMES:
        per_class_rows.append({
            'model_key': model_key,
            'class': class_name,
            'precision': report[class_name]['precision'],
            'recall': report[class_name]['recall'],
            'f1_score': report[class_name]['f1-score'],
            'support': report[class_name]['support'],
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_path = REPORT_DIR / f'{MODEL_KEY}_tflite_per_class_metrics.csv'
per_class_df.to_csv(per_class_path, index=False)
print(f'Saved per-class metrics: {per_class_path}')
display(per_class_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for model_key in tflite_summary_df['model_key'].dropna().tolist():
    model_preds = tflite_predictions_df[tflite_predictions_df['model_key'] == model_key]
    if model_preds.empty:
        continue
    cm = confusion_matrix(model_preds['true_label'], model_preds['predicted_label'], labels=CLASS_NAMES)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
    plt.title(model_key)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    fig_path = REPORT_DIR / f'confusion_matrix_{model_key}.png'
    plt.savefig(fig_path, dpi=160)
    plt.show()
    print(f'Saved: {fig_path}')

## Result Artifacts

The key outputs are saved under:

`/content/drive/MyDrive/shrimp_disease_images/convnext_tiny_in22k_baseline`

Most important files:

- PyTorch training history: `reports/convnext_tiny_in22k_training_history.csv`
- PyTorch held-out summary: `reports/convnext_tiny_in22k_pytorch_test_summary.csv`
- ONNX model: `exports/convnext_tiny_in22k.onnx`
- TFLite model: `tflite/convnext_tiny_in22k_float32.tflite`
- TFLite held-out summary: `reports/convnext_tiny_in22k_tflite_test_summary.csv`
- TFLite per-class metrics: `reports/convnext_tiny_in22k_tflite_per_class_metrics.csv`
- TFLite predictions: `reports/convnext_tiny_in22k_tflite_test_predictions.csv`